<img align='left' alt='ESO Logo' src='http://archive.eso.org/i/esologo.png'> 
# &nbsp;How to download data 
<br>

This section of the ["ESO Science Archive Programmatic: HOWTOs"](http://archive.eso.org/programmatic/HOWTO/) shows how to programmatically download ESO data, either anonymously (for public data) or with authentication (for proprietary data), using Python.

_**Usage**: You can access this file either as a static HTML page [(download it here)](http://archive.eso.org/programmatic/HOWTO/jupyter/ESO_How_to_download_data.html), or as an interactive jupyter notebook [(download it here)](http://archive.eso.org/programmatic/HOWTO/jupyter/ESO_How_to_download_data.ipynb) which you can download and run on your machine [(instructions)](https://jupyter.org/install). To interact with the jupyter notebook: move up and down the various cells using the arrow keys, execute the code by pressing CTRL+ENTER; you can also modify the code and execute it at will._

<hr>

Let's start by setting up the python modules:

In [ ]:
import os
import sys

import requests
import cgi
import json

import getpass

Let's define a couple of utility functions, useful to write the files on disk using the ESO file name (provided in the response http header, via the Content-Disposition field.

In [ ]:
def getDispositionFilename(response):
    """Get the filename from the Content-Disposition in the response's http header"""
    contentdisposition = response.headers.get('Content-Disposition')
    if contentdisposition == None:
        return None
    value, params = cgi.parse_header(contentdisposition)
    filename = params['filename']
    return filename


def writeFile(response, dirname='data'):
    """Write on disk the retrieved file"""
    if response.status_code == 200:
        # The ESO filename can be found in the response header
        filename = getDispositionFilename(response)
        os.makedirs(dirname, exist_ok=True)
        filepath = os.path.join(dirname, filename)
        # Let's write on disk the downloaded FITS spectrum using the ESO filename:
        with open(filepath, 'wb') as f:
            f.write(response.content)
        return filepath

## How to retrieve a file anonymously

Without the need to authenticate, any user can anonymously download public files, that is, files that are out of the proprietary period (of usually one year from the moment the observation takes place).

In [ ]:
file_url = 'https://dataportal.eso.org/dataportal_new/file/ADP.2016-11-17T12:51:01.877'

response = requests.get(file_url)
filename = writeFile(response)
if filename:
    print('Saved file: %s' % (filename))
else:
    print('Could not get file (status: %d)' % (response.status_code))

## How to retrieve a file with authentication

If the files you need to retrieve are under proprietary period, you can access them only if you have the rights to do so, that is, if you are the principal investigator [PI] of the observing program the files belong to, or one of his/her delegates. In this case, you certainly got already a (free) user account at the [ESO User Portal](https://www.eso.org/UserPortal).

Before downloading the file you have to authenticate and get a token. Here is the method that, given your ESO credentials (username and password), returns the token.

In [ ]:
def getToken(username, password):
    """Token based authentication to ESO: provide username and password to receive back a JSON Web Token."""
    if username == None or password == None:
        return None
    token_url = 'https://www.eso.org/sso/oidc/token'
    token = None
    try:
        response = requests.get(
            token_url,
            params={
                'response_type': 'id_token token',
                'grant_type': 'password',
                'client_id': 'clientid',
                'username': username,
                'password': password,
            },
        )
        token_response = json.loads(response.content)
        token = token_response['id_token'] + '=='
    except NameError as e:
        print(e)
    except:
        print('*** AUTHENTICATION ERROR: Invalid credentials provided for username %s' % (username))

    return token

Once you get the token, you need to add it to the HTTP header before HTTP-getting the file. Let's see how:

In [ ]:
# Suppose the file you want to download is accessible via the following link
# (please change the identifier (ADP.2020-03-24T1:45:21.866) to one of your proprietary files):

file_url = 'https://dataportal.eso.org/dataportal_new/file/ADP.2020-03-24T10:45:21.866'

# If you have not modified the identifier of that file,
# likely you won't be authorised to download the file.
# Expect a failure "Could not get file (status: 401)" in that case.

# Let's get the token, by inputting your credentials:
username = input('Type your ESO username: ')
password = getpass.getpass(prompt="%s user's password: " % (username), stream=None)
token = getToken(username, password)

# With successful authentication you get a valid token,
# which needs to be added to the HTTP header of your GET request,
# as a Bearer:

headers = None
if token != None:
    headers = {'Authorization': 'Bearer ' + token}
    response = requests.get(file_url, headers=headers)
    filename = writeFile(response)
    if filename:
        print('Saved file: %s' % (filename))
    else:
        print('Could not get file (status: %d)' % (response.status_code))
else:
    print('Could not authenticate')

In [ ]:
import pyvo
from pyvo.dal import tap
from pyvo.auth.authsession import AuthSession

In [ ]:
import eso_programmatic as eso

TAP_URL = 'http://archive.eso.org/tap_obs'

In [ ]:
# Prompt for user's credentials and get a token
import getpass

username = input('Type your ESO username: ')
password = getpass.getpass(prompt=f"{username}'s password: ", stream=None)

token = eso.getToken(username, password)
if token != None:
    print('token: ' + token)
else:
    sys.exit(-1)

In [ ]:
session = requests.Session()
session.headers['Authorization'] = 'Bearer ' + token

# Initialise a tap service for authorised queries
# passing the created "tokenised" session
# Remember: passing a non tokenised-session, or no session at all,
# will result in tap performing anonymous queries:
# none of your permissions will be used, hence the queryies will run faster,
# and you will not be able to find any file with protected metadata.

tap = pyvo.dal.TAPService(TAP_URL, session=session)

# for comparison, use:
# tap = pyvo.dal.TAPService(TAP_URL)
# to execute your queries anonymously

In [ ]:
# define the query you want to run, e.g.:
query = "select top 2 * from dbo.raw where dp_cat='SCIENCE' and prog_id = 'your-protected-observing-run' "

# well, in this example we use a non-protected run,
# but please pretend it is actually a protected one given the purpose of this notebook!

# let's consider only 2 of its science frames:
query = "select top 10 * from dbo.raw where dp_cat='SCIENCE' and prog_id = '116.28N5.001' "


results = None

# define a job that will run the query asynchronously
job = tap.submit_job(query)

# extending the maximum duration of the job to 300s (default 60 seconds)
job.execution_duration = 300  # max allowed: 3600s

# job initially is in phase PENDING; you need to run it and wait for completion:
job.run()

try:
    job.wait(phases=['COMPLETED', 'ERROR', 'ABORTED'], timeout=600.0)
except pyvo.DALServiceError:
    print('Exception on JOB {id}: {status}'.format(id=job.job_id, status=job.phase))

print('Job: %s %s' % (job.job_id, job.phase))

if job.phase == 'COMPLETED':
    # When the job has completed, the results can be fetched:
    results = job.fetch_result()

# the job can be deleted (always a good practice to release the disk space on the ESO servers)
job.delete()

# Let's print the results to examine the content:
# check out the access_url and the datalink_url
if results:
    print('query results:')
    eso.printTableTransposedByTheRecord(results.to_table())
else:
    print('!' * 42)
    print('!                                        !')
    print('!       No results could be found.       !')
    print('!       ? Perhaps no permissions ?       !')
    print('!       Aborting here.                   !')
    print('!                                        !')
    print('!' * 42)
    quit()

## 3 Downloading the selected science files using their access_url

In [ ]:
# The access_url field of the dbo.raw table
# provides the link that can be used to download the file

# Here we pass that link together with your session
# to the downloadURL method of the eso_programmatic.py module
# (similarly to the authorised queries, if no session is passed,
#  downloadURL will attempt to download the file anonymously)

print('Start downloading...')
for raw in results:
    access_url = raw['access_url']  # the access_url is the link to the raw file
    status, filepath = eso.downloadURL(access_url, session=session, dirname='/apophis/tlister/VLT/FORS/116.28N5.001/')
    if status == 200:
        print(f'      RAW: {filepath} downloaded  ')
    else:
        print('ERROR RAW: {filepath} NOT DOWNLOADED (http status:{status})')

## 4 Finding and downloading the associated calibration reference files

### 4.1 Find the link to the associated calibration reference files (using DataLink)

The datalink_url field of the dbo.raw table provides you the link that can be used to find files associated to the selected science frame.

In [ ]:
# A python datalink object is created running
# the pyvo DataLinkResults.from_result_url() method onto the datalink_url.

# When dealing with files whose metadata are protected, we need to be authorised:
# for that we need to pass to the from_result_url() also the above-created python requests session.

# For the sake of this example, let's just consider the first science raw frame:
first_record = results[0]
datalink_url = first_record['datalink_url']

datalink = pyvo.dal.adhoc.DatalinkResults.from_result_url(datalink_url, session=session)

# The resulting datalink object contains the table of files associated
# to SPHER.2016-09-26T03:04:09.308
# Note: Were this input file a metadata protected file (it is not, but suppose...),
# and had you not passed your session, or had you no permission to see this file,
# DataLink would have given you back only a laconic table with the message
# that that you do not have access permissions or that the file does not exist.

# let's print the resulting datalink table:
eso.printTableTransposedByTheRecord(datalink.to_table())

In [ ]:
# Let's get the link to the processed calibration files (raw2master)

semantics = 'http://archive.eso.org/rdf/datalink/eso#calSelector_raw2master'

raw2master_url = next(datalink.bysemantics(semantics)).access_url

# which returns the calSelector (see next box) link:
# https://archive.eso.org/calselector/v1/associations?dp_id=\
# SPHER.2016-09-26T03:04:09.308&mode=Raw2Master&responseformat=votable

In [ ]:
# Don't forget to pass your session in case the science file has protected metadata!

associated_calib_files = pyvo.dal.adhoc.DatalinkResults.from_result_url(raw2master_url, session=session)

eso.printTableTransposedByTheRecord(associated_calib_files.to_table())

# create and use a mask to get only the #calibration entries,
# given that other entries, like #this or ...#sibiling_raw, could be present:
calibrator_mask = associated_calib_files['semantics'] == '#calibration'
calib_urls = associated_calib_files.to_table()[calibrator_mask]['access_url', 'eso_category']

# eso.printTableTransposedByTheRecord(calib_urls)

### 4.2 Getting the list of processed calibration reference files (using calSelector and DataLink)
The automatic selection of calibration files (raw or processed) is performed by the above-mentioned calSelector service, exposed also programmatically.

One of the calSelector interfaces (the responseformat=votable param must be present), is fully compatible with the datalink VO protocol. This means that the same pyvo DatalinkResults.from_result_url() method can be used, e.g., to get the list of associated raw2master files.

In [ ]:
# Don't forget to pass your session in case the science file has protected metadata!

associated_calib_files = pyvo.dal.adhoc.DatalinkResults.from_result_url(raw2master_url, session=session)

eso.printTableTransposedByTheRecord(associated_calib_files.to_table())

# create and use a mask to get only the #calibration entries,
# given that other entries, like #this or ...#sibiling_raw, could be present:
calibrator_mask = associated_calib_files['semantics'] == '#calibration'
calib_urls = associated_calib_files.to_table()[calibrator_mask]['access_url', 'eso_category']

# eso.printTableTransposedByTheRecord(calib_urls)

### 4.2.1 Check calibration cascade qualities

Check if calibration cascade is complete, if it is certified, and if it is actually for processed calib files

In [ ]:
# Given the above list of "associated_calib_files"
# and knowing that we requested...
mode_requested = 'raw2master'

# ... let's print out some important info and warnings on the received calibration cascade:
# - is the cascade complete?
# - is the cascade certified?
# - has the cascade being generated for the mode you requested (processed calibrations) or not?

# That info is embedded in the description field of the #this record.
# We use the printCalselectorInfo of the eso_programmatic.py to parse/make sense of it.

this_description = next(associated_calib_files.bysemantics('#this')).description

alert, mode_warning, certified_warning = eso.printCalselectorInfo(this_description, mode_requested)

if alert != '':
    print('%s' % (alert))
if mode_warning != '':
    print('%s' % (mode_warning))
if certified_warning != '':
    print('%s' % (certified_warning))

question = None
answer = None
if len(calib_urls):
    print()
    if alert or mode_warning or certified_warning:
        question = 'Given the above warning(s), do you still want to download these %d calib files [y/n]? ' % (
            len(calib_urls)
        )
    else:
        question = 'No warnings reported, do you want to download these %d calib files [y/n]? ' % (len(calib_urls))

while answer != 'y' and answer != 'n':
    answer = input(question)

## 4.3 Downloading the calibration reference files

To download the calibration files we use again the downloadURL method of the eso_programmatic.py module.

All ESO calibration files are open to the public, hence there is no need to pass your token/session.


In [ ]:
if answer == 'y':
    print('Downloading the %d calibration reference files...' % (len(calib_urls)))

    i_calib = 0
    for url, category in calib_urls:
        i_calib += 1
        status, filename = eso.downloadURL(url, dirname='/apophis/tlister/VLT/FORS/116.28N5.001/')
        if status == 200:
            print('    CALIB: %4d/%d dp_id: %s (%s) downloaded' % (i_calib, len(calib_urls), filename, category))
        else:
            print(
                '    CALIB: %4d/%d dp_id: %s (%s) NOT DOWNLOADED (http status:%d)'
                % (i_calib, len(calib_urls), filename, category, status)
            )

In [ ]:
association_tree_semantics = 'http://archive.eso.org/rdf/datalink/eso#calSelector_raw2master'

# Notice that the datalink service and the calselector service use the same semantics
# to indicate two different things:
# - in datalink: it points to the distinct list of calibration reference files (responseformat=votable);
#                its eso_category is not defined
# - in calselector: it points to the calibration cascade description (format still XML but not votable);
#                its eso_category is set to "ASSOCIATION_TREE"

association_tree_mask = associated_calib_files['semantics'] == association_tree_semantics
association_tree = associated_calib_files.to_table()[association_tree_mask]['access_url', 'eso_category']

for url, category in association_tree:
    # the url points to the calselector service, which, for metadata protected files, needs a tokenised-session
    status, filename = eso.downloadURL(url, session=session)
    print(url)
    if status == 200:
        print('  Association tree: %s (%s) downloaded' % (filename, category))
    else:
        print('  Association tree: %s (%s) NOT DOWNLOADED (http status:%d)' % (filename, category, status))